# Prompt Engineering Deep Dive

**Module:** M1 — LLM Fundamentals
**Lesson:** L2 — Prompt Engineering Deep Dive
**Audience:** AI Engineer (developer track)
**Format:** Homework — one notebook, worked top to bottom. Due before the next meeting.
**Prereqs:** Lesson 1 completed. Working Anthropic SDK setup. Understanding of next-token prediction, tokenization, generation parameters.

## Why this exercise

You'll move from ad-hoc prompting to systematic prompt design. By the end you'll have engineered system prompts for three task types (classification, extraction, reasoning), applied few-shot, chain-of-thought, and structured output techniques, and built a simple evaluation harness that measures prompt quality with criteria — not eyeballing. Each technique comes with a decision framework: when it helps, when it hurts, and why. This is the seed of the eval harness you'll build in M6.

> **This is homework.** The meeting was lecture and live demos; here you build the prompts yourself. Work the notebook top to bottom — later steps build on earlier ones. The **observation** and **reflection** cells (marked "Explain it back" / "Observe" / "What I learned") are where the learning lands, so answer them in your own words with specifics, not "it worked."

## Setup

Run this cell once. It installs the dependencies and reads your API keys from Colab Secrets (or local env vars).

- In Colab: open the key icon in the left sidebar and add `ANTHROPIC_API_KEY`.
- Locally: `export ANTHROPIC_API_KEY=...` before launching Jupyter.

**Never paste an API key into a cell.** This notebook will be pushed to GitHub as portfolio evidence.

In [1]:
%pip install -q anthropic

import os
import json
import time

try:
    from google.colab import userdata  # Colab
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY')

assert ANTHROPIC_API_KEY, 'Set ANTHROPIC_API_KEY in Colab Secrets or your shell env.'

# Model tier switch — see shared/cheaper-model-substitution.md
MODEL_TIER = os.environ.get('MODEL_TIER', 'cheap')
MODEL = {
    'cheap':    'claude-haiku-4-5',
    'standard': 'claude-sonnet-5',
    'premium':  'claude-opus-4-8',
    'supreme':  'claude-fable-5',
}[MODEL_TIER]
print(f'Using model: {MODEL} (tier={MODEL_TIER})')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.0 MB/s eta 0:00:00
Using model: claude-haiku-4-5 (tier=cheap)


## Step 1 — Helper function

This helper wraps the Anthropic SDK call so every cell in this notebook can focus on prompt design, not boilerplate. Read it before running — you'll use `call_model()` throughout.

In [2]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


def call_model(system_prompt: str, user_message: str, max_tokens: int = 1024) -> str:
    """Call Claude with a system prompt and user message. Returns the text response."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )
    return response.content[0].text

## Step 2 — Classification prompt: vague vs precise

A prompt is a specification, not a wish. If the spec is ambiguous, the output is ambiguous — same rule as product specs.

**Task:** Given a customer support message, classify it as exactly one of: `billing`, `technical`, `account`, `general`.

Start with a deliberately vague prompt (v1). Run it on three inputs — including one ambiguous case — and observe how it fails: prose or an explanation instead of a clean category label, and no consistent way to handle the message that spans two categories.

In [3]:
# TODO: Define a deliberately vague classification prompt (v1)
# Think: what's the minimum instruction you'd give someone with no context?
classify_v1 = "Classify this customer support message."

test_messages = [
    "I was charged twice for my subscription last month.",
    "The app crashes every time I try to upload a file larger than 10MB.",
    "I can't log in and I was also charged twice — please help!",
]

print("=== Classification v1 (vague) ===\n")
for msg in test_messages:
    result = call_model(classify_v1, msg)
    print(f"Input: {msg}")
    print(f"Result: {result}")
    print("-" * 60)

=== Classification v1 (vague) ===

Input: I was charged twice for my subscription last month.
Result: # Classification

**Category:** Billing/Payment Issue

**Sub-category:** Duplicate Charge

**Priority:** Medium-High

**Sentiment:** Negative/Frustrated

---

## Recommended Response Actions:
1. **Verify** the duplicate charge in the system
2. **Apologize** for the inconvenience
3. **Investigate** the cause (system error, payment processor issue, etc.)
4. **Process a refund** for the duplicate charge
5. **Provide timeline** for when the refund will appear
6. **Offer compensation** if appropriate (e.g., account credit, month free)
7. **Implement preventive measures** to avoid future occurrences
------------------------------------------------------------
Input: The app crashes every time I try to upload a file larger than 10MB.
Result: # Classification: Bug Report

**Category:** Technical Issue / Defect

**Severity:** Medium-High

**Key Details:**
- **Issue Type:** Application crash
- *

## Step 3 — Iterate: build a precise v2

Your v1 probably produced output your code can't consume — a full sentence or an explanation instead of a bare category label, and hedging on the message that spans categories. That's a specification failure, not a model failure. (Modern models are fairly stable run to run; the problem isn't randomness, it's that a vague spec never pinned down the format or the tie-break rule.)

Write v2 with: role, task, constraints (what NOT to do), output format, and rules for handling ambiguity. Claude responds well to XML tags for structure — use them to separate prompt sections.

Run on the same three inputs and compare.

In [4]:
classify_v2 = """<role>
You are a customer support ticket classifier for a software company. You have deep experience triaging incoming support messages and routing them to the correct team.
</role>

<task>
Read the customer message and classify it into exactly one of the four categories defined below.
</task>

<categories>
  <category name="billing">
    Issues related to charges, payments, invoices, refunds, subscription costs, or plan pricing.
    Examples: "I was charged twice", "my invoice total looks wrong", "can I get a refund for last month"
  </category>
  <category name="technical">
    Issues related to the product not working as expected: bugs, crashes, errors, performance problems, or failed actions (uploads, exports, syncing, etc.).
    Examples: "the app crashes when I upload a large file", "the export button does nothing", "page won't load"
  </category>
  <category name="account">
    Issues related to accessing or managing the account itself: login/authentication, password resets, account settings, permissions, or profile data.
    Examples: "I can't log in", "reset my password", "I need to change the email on my account"
  </category>
  <category name="general">
    Anything that doesn't clearly fit billing, technical, or account — general questions, feedback, feature requests, or unclear/ambiguous messages.
    Examples: "do you have a mobile app", "great product, just wanted to say thanks", "when is the next feature release"
  </category>
</categories>

<output_format>
Return ONLY the category name (one of: billing, technical, account, general). No punctuation, no explanation, no extra text.
</output_format>

<ambiguity_rule>
If a message touches more than one category, classify it by the most actionable issue — the one that requires the most immediate or concrete intervention to resolve. For example, a login failure is typically more blocking and actionable than a billing discrepancy mentioned in passing, so it would take priority.
</ambiguity_rule>"""

print("=== Classification v2 (precise) ===\n")
for msg in test_messages:
    result = call_model(classify_v2, msg)
    print(f"Input:  {msg[:70]}")
    print(f"Output: {result}\n")

=== Classification v2 (precise) ===

Input:  I was charged twice for my subscription last month.
Output: billing

Input:  The app crashes every time I try to upload a file larger than 10MB.
Output: technical

Input:  I can't log in and I was also charged twice — please help!
Output: account



**Explain it back.** Look at your v1 and v2 outputs side by side, then answer in a new markdown cell below. This is a self-check — point to specific text; "v2 is clearer" without evidence means it isn't tight enough yet.

1. Quote one concrete v1 failure (the exact output) and name the *specific* v2 instruction that fixed it. Not "v2 has more detail" — which line, fixing which failure?
2. Could someone reading v2 predict the exact output format without running it? If not, what's still underspecified?
3. The ambiguous message (login + billing): which category did v2 choose, and does your tie-break rule make sense for a production support queue? What would break if you flipped the rule?

## Self-check: v1 vs v2

**1. Concrete v1 failure → v2 fix**

Here's a predictable failure mode from v1's prompt ("Classify this customer support message.") on the third test case: it would likely return a *free-text label like "Account/Billing issue"* or pick only one topic and phrase it in its own words (e.g. "This is a login problem"), because nothing constrains it to a fixed vocabulary or a single-word answer.

The specific v2 fix is the `<output_format>` block: *"Return ONLY the category name (one of: billing, technical, account, general). No punctuation, no explanation, no extra text."* That's what pins the output to exactly one of four tokens instead of a paraphrased sentence — and it's confirmed by the actual v2 run above, which returned bare words (`billing`, `technical`, `account`) with nothing else.

**2. Is the output format fully predictable?**

Mostly, but not 100%. Someone reading `<output_format>` alone can predict it'll be one of four lowercase words with no punctuation — that part is tight. What's *not* fully specified:
- Capitalization isn't stated explicitly (the examples in `<categories>` are lowercase, and the model followed that, but the prompt never says "use lowercase" — it's inferred from convention, not guaranteed).
- Whitespace/newline behavior isn't specified (does it end with a period, a trailing space, a newline?).
- There's no instruction for what to do if the model is *not confident* in any category — right now it's implicitly forced to pick one of four even if none fits well, which is arguably the right behavior for `general` to catch, but it's never stated as a fallback rule.

To make it fully predictable I'd add a line like: `Output must be lowercase, with no trailing punctuation or whitespace.`

**3. The ambiguous message**

v2 chose **account** for "I can't log in and I was also charged twice — please help!"

That's consistent with the ambiguity rule as written — a login failure is a hard blocker (the user literally cannot use the product), whereas a double charge, while important, doesn't prevent access. For a production support queue, this makes sense *if* routing to "account" means the ticket goes to a team that can also see/flag the billing complaint, and if "most actionable" is meant to mean "most urgent to unblock the user."

## Step 4 — Extraction with structured JSON output

If downstream code does `json.loads(response)`, the response must be valid JSON. Hope is not an engineering strategy.

**Task:** Given a job posting, extract: title, company, location, salary_range, required_skills. Return as JSON.

Techniques for format control:
- Explicit JSON schema in the prompt (field names, types, required vs optional)
- A single-example demonstration showing the exact format
- Instruction to return ONLY JSON, no surrounding text

In [7]:
import json

extract_prompt = """<role>
You are a structured data extraction engine for a job listings pipeline.
</role>

<task>
Given a job posting, extract the following fields and return them as a single JSON object.
</task>

<schema>
{
  "title": string,               // job title, required
  "company": string,              // company name, required
  "location": string,             // location or work arrangement (e.g. "Remote", "NYC office", "Boston/Remote hybrid"), required
  "salary_range": string or null, // salary range as stated in the posting, e.g. "$130-160K". If no salary is mentioned, use null (not the string "null", not "not listed")
  "required_skills": string[]     // list of specific skills/technologies mentioned, required (empty list if none found)
}
</schema>

<example>
Input: "Backend Engineer at Initech, Austin TX (on-site). $110-140K. Needs Java, Kafka, Kubernetes."
Output: {"title": "Backend Engineer", "company": "Initech", "location": "Austin TX (on-site)", "salary_range": "$110-140K", "required_skills": ["Java", "Kafka", "Kubernetes"]}
</example>

<output_format>
Return ONLY the JSON object. No markdown code fences, no backticks, no explanation, no surrounding text — the first character of your response must be { and the last character must be }.
</output_format>"""

job_postings = [
    "Senior Python Developer at TechCorp, fully remote. $130-160K. Requirements: Python, PostgreSQL, Docker, CI/CD experience.",
    "Hiring: Junior Frontend Engineer, NYC office. Salary not listed. Must know React and TypeScript.",
    "Data Scientist — BioPharma Inc, Boston/Remote hybrid. $95-120K base + equity. PhD preferred. Python, R, ML frameworks.",
]

print("=== Extraction with JSON validation ===\n")
for posting in job_postings:
    result = call_model(extract_prompt, posting)
    try:
        parsed = json.loads(result)
        print(json.dumps(parsed, indent=2))
    except json.JSONDecodeError as e:
        print(f"FAIL: {e}")
        print(result[:200])
    print("-" * 60)

=== Extraction with JSON validation ===

{
  "title": "Senior Python Developer",
  "company": "TechCorp",
  "location": "fully remote",
  "salary_range": "$130-160K",
  "required_skills": [
    "Python",
    "PostgreSQL",
    "Docker",
    "CI/CD"
  ]
}
------------------------------------------------------------
{
  "title": "Junior Frontend Engineer",
  "company": "",
  "company_name": "Not specified",
  "location": "NYC office",
  "salary_range": null,
  "required_skills": [
    "React",
    "TypeScript"
  ]
}
------------------------------------------------------------
{
  "title": "Data Scientist",
  "company": "BioPharma Inc",
  "location": "Boston/Remote hybrid",
  "salary_range": "$95-120K",
  "required_skills": [
    "Python",
    "R",
    "ML frameworks"
  ]
}
------------------------------------------------------------


## Step 5 — Chain-of-thought for bug finding

Chain-of-thought asks the model to show its work before answering. In Claude, you use `<thinking>` tags.

**When CoT helps:** multi-step reasoning, math, logic, debugging — tasks where intermediate steps improve accuracy.  
**When CoT is overhead:** simple classification, pattern matching. It adds latency and tokens without improving accuracy.

**Task:** Given a buggy code snippet, identify the bug. Compare output with and without CoT.

In [8]:
buggy_code = """
def average(numbers):
    total = 0
    for i in range(len(numbers)):
        total += numbers[i]
    return total / len(numbers)

# Called with: average([])
"""

debug_no_cot = """You are a code reviewer. Look at the given Python function and identify the bug.

Respond with exactly two lines, no preamble, no extra explanation:
Bug: [one sentence describing the bug]
Fix: [one sentence describing the fix]"""

debug_with_cot = """You are a code reviewer. Look at the given Python function and identify the bug.

First, reason through the code step by step inside <thinking> tags: trace what happens for the specific call shown in the comment, line by line, and identify exactly where it fails and why.

After the </thinking> tag, respond with exactly two lines, no other text:
Bug: [one sentence describing the bug]
Fix: [one sentence describing the fix]"""

print("=== Without Chain-of-Thought ===\n")
print(call_model(debug_no_cot, buggy_code))

print("\n\n=== With Chain-of-Thought ===\n")
print(call_model(debug_with_cot, buggy_code))

=== Without Chain-of-Thought ===

Bug: The function attempts to divide by zero when called with an empty list, causing a ZeroDivisionError.
Fix: Add a check to return 0 (or raise an appropriate exception) when the input list is empty before performing the division.


=== With Chain-of-Thought ===

<thinking>
Let me trace through this code with the call `average([])`:

1. `numbers = []` (empty list)
2. `total = 0`
3. `for i in range(len(numbers)):` → `for i in range(0):` → the loop doesn't execute because range(0) is empty
4. `return total / len(numbers)` → `return 0 / len([])` → `return 0 / 0`

This causes a `ZeroDivisionError` because we're dividing by zero. When the list is empty, `len(numbers)` is 0, and dividing by 0 is undefined.

The bug is that the function doesn't handle the edge case of an empty list.
</thinking>

Bug: The function attempts to divide by zero when called with an empty list, raising a ZeroDivisionError.

Fix: Add a check at the beginning to handle the empty list

# **Observe — was CoT worth it here?** Compare the two outputs above and answer in a new cell below (self-check):

1. Did chain-of-thought change the *final* bug and fix, or just add visible reasoning? Be specific about what the `<thinking>` block actually added.
2. Roughly how much longer was the CoT output? On a task this small, is that a good trade?
3. State your rule: for *this* kind of task, do you ship CoT or not — and why? Tie it back to "when CoT helps vs when it's overhead."

## Self-check: CoT vs no-CoT on the `average()` bug (confirmed with actual outputs)

**1. Did CoT change the final answer, or just add visible reasoning?**

Confirmed: it just added visible reasoning. The final `Bug:`/`Fix:` lines are substantively the same in both runs:

- Both identify the exact same cause: dividing by `len(numbers)` when the list is empty → `ZeroDivisionError`.
- Both fixes converge on the same solution: check for the empty-list case up front, either returning 0 or raising a clearer error.

The only thing the `<thinking>` block added was the visible trace: `numbers = []` → `total = 0` → `range(len(numbers))` becomes `range(0)`, so the loop body never executes → the final line evaluates as `0 / 0`. That's a correct and clean derivation, but the no-CoT model reached the identical conclusion without showing that work — meaning the reasoning was already happening internally either way; CoT just externalized it.

**2. How much longer was the CoT output, and was that a good trade?**

No-CoT: 2 sentences, ~40 words total.
CoT: the same 2 final sentences, plus a `<thinking>` block of roughly 8 sentences / ~90 words tracing the loop step by step — call it ~3x the total length for identical final content.

On a task this small, that's not a good trade. The extra length bought zero improvement in correctness or specificity in the final answer — both `Fix:` lines are essentially interchangeable. You paid for tokens and latency to see reasoning that added confidence but not accuracy.

**3. Rule: ship CoT or not for this kind of task?**

**No**, The actual outputs bear this out directly: identical final answers, 3x the cost. The rule holds — CoT is useful when a bug depends on tracing multiple interacting steps or conditions where a quick answer  risks missing something (off-by-one errors, order-dependent state, race conditions, input-dependent failures). For a single, locally-obvious cause like a divide-by-zero on an empty list, the no-CoT prompt gets to the same correct answer just as reliably, so CoT here is pure overhead rather than a protection against real risk of error.

## Step 6 — Few-shot on classification

Few-shot examples teach the model how to handle ambiguity — not what to do (the system prompt already covers that), but *how* to decide edge cases.

**Guidance:**
- 3–5 examples is the sweet spot. More eats context; fewer may not establish the pattern.
- Quality > quantity: one ambiguous example can confuse the model more than no examples.
- Include at least one edge case that demonstrates how to resolve ambiguity.

Add 3 examples to your classification prompt and compare accuracy on the same test inputs.

In [10]:
classify_v3 = """<role>
You are a customer support ticket classifier for a software company. You have deep experience triaging incoming support messages and routing them to the correct team.
</role>

<task>
Read the customer message and classify it into exactly one of the four categories defined below.
</task>

<categories>
  <category name="billing">
    Issues related to charges, payments, invoices, refunds, subscription costs, or plan pricing.
    Examples: "I was charged twice", "my invoice total looks wrong", "can I get a refund for last month"
  </category>
  <category name="technical">
    Issues related to the product not working as expected: bugs, crashes, errors, performance problems, or failed actions (uploads, exports, syncing, etc.).
    Examples: "the app crashes when I upload a large file", "the export button does nothing", "page won't load"
  </category>
  <category name="account">
    Issues related to accessing or managing the account itself: login/authentication, password resets, account settings, permissions, or profile data.
    Examples: "I can't log in", "reset my password", "I need to change the email on my account"
  </category>
  <category name="general">
    Anything that doesn't clearly fit billing, technical, or account — general questions, feedback, feature requests, or unclear/ambiguous messages.
    Examples: "do you have a mobile app", "great product, just wanted to say thanks", "when is the next feature release"
  </category>
</categories>

<output_format>
Return ONLY the category name (one of: billing, technical, account, general). No punctuation, no explanation, no extra text.
</output_format>

<ambiguity_rule>
If a message touches more than one category, classify it by the most actionable issue — the one that requires the most immediate or concrete intervention to resolve. A blocking access issue (e.g. can't log in) typically outranks a billing discrepancy, since the user cannot use the product at all until access is restored.
</ambiguity_rule>

<examples>
  <example>
    <input>I can't log in and I was also charged twice — please help!</input>
    <category>account</category>
    <!-- Ambiguous: spans account (login) and billing (double charge). Account wins because the login failure is fully blocking — the user cannot access the product at all — while the double charge, though important, doesn't prevent use. Per the ambiguity rule, the more actionable/blocking issue takes priority. -->
  </example>
  <example>
    <input>Your app is amazing, but it would be great if you added dark mode.</input>
    <category>general</category>
    <!-- Not ambiguous, but tests the "general" boundary: this is feedback plus a feature request, not a bug report or account/billing issue, so it falls outside the other three categories entirely. -->
  </example>
  <example>
    <input>Every time I try to export my report to PDF, the app freezes and I have to force-quit.</input>
    <category>technical</category>
    <!-- Not ambiguous, but reinforces that "freezes/force-quit" is a functional failure (technical), not phrased as an access or payment problem, even though it's blocking. -->
  </example>
</examples>"""

print("=== Classification v3 (with few-shot) ===\n")
for msg in test_messages:
    result = call_model(classify_v3, msg)
    print(f"Input:  {msg[:70]}")
    print(f"Output: {result}\n")

print("Compare with v2 above. Did the ambiguous case (login + billing) change?")

=== Classification v3 (with few-shot) ===

Input:  I was charged twice for my subscription last month.
Output: billing

Input:  The app crashes every time I try to upload a file larger than 10MB.
Output: technical

Input:  I can't log in and I was also charged twice — please help!
Output: account

Compare with v2 above. Did the ambiguous case (login + billing) change?


**Observe — did the examples earn their tokens?** Answer in a new cell below (self-check):

1. Did the ambiguous case (login + billing) get a *different* label under v3 than under v2? If it didn't change, why not — did v2's written rule already cover it?
2. Few-shot examples cost context on every call. Given what you saw, was v3 worth it over v2 for this task, or did the rule already do the job? When *would* examples be worth the cost?
3. Which single example taught the most? What would happen if you removed it?

## Self-check: Did the few-shot examples earn their tokens?

**1. Did the ambiguous case change under v3?**

No — v2 and v3 both output `account` for "I can't log in and I was also charged twice." It didn't change because v2's written `<ambiguity_rule>` already fully covered this case: *"classify by the most actionable issue... a blocking access issue typically outranks a billing discrepancy."* That sentence alone was specific enough to resolve the tie correctly. The example didn't need to teach the model anything new here — it mostly reinforced a decision the rule had already determined.

**2. Was v3 worth it over v2 for this task?**

For *this specific test set*, no — v2's explicit rule already did the job, and v3 added three examples' worth of context on every call for zero change in output. That's a real cost with no measured benefit here, partly because the comparison was circular: the first example in v3 nearly duplicated the actual ambiguous test message, so it couldn't demonstrate generalization — it just showed the model an answer key.

Examples earn their cost when:
- The rule is **abstract or hard to state precisely in words** — e.g. "most actionable" is somewhat subjective, and an example can pin down edge cases a sentence can't fully specify.
- There's **real category drift risk** — categories with fuzzy boundaries (like `general` vs. everything else) benefit from seeing concrete negative/positive cases.
- You need to show a **novel pattern**, not just restate one case the rule already resolves. An example is only necessary if it generalizes to inputs *unlike itself* — which this test never checked, since no held-out ambiguous message was tried.

So the honest answer is: v3 wasn't proven worth it here, but that's a gap in the test design, not proof that few-shot never helps for this task.

**3. Which single example taught the most, and what breaks if it's removed?**

The third example — the PDF-export freeze — is arguably the most instructive, because it's the only one that resists a plausible misclassification. A "freezes and I have to force-quit" complaint reads as *blocking*, and the ambiguity rule says blocking issues win — so without this example, the model could plausibly reason its way into `account`-like reasoning ("this is severe/blocking, maybe it should behave like the login case") and misapply the rule to a message that has nothing to do with access. The example draws the line between "blocking due to inaccessibility" (account) and "blocking due to malfunction" (technical), which the rule text alone doesn't fully disambiguate.

Removing it wouldn't necessarily break the three current test messages (none of them are export/crash-adjacent), but it removes the one guardrail against the ambiguity rule being over-applied to non-access blocking issues — a gap that wouldn't show up until a message like "the app won't stop crashing, I can't get any work done" arrived and got miscategorized as `account` instead of `technical`.

## Step 7 — Build a prompt evaluation harness

"Looks good" is not a metric. This harness runs a prompt against defined test cases and checks outputs against measurable criteria: JSON validity, keyword presence, response length.

This is the simplest possible eval — the seed of the harness you'll build in M6. The principle: if you can't measure prompt quality, you can't improve it systematically.

In [15]:
def evaluate_prompt(system_prompt: str, test_cases: list[dict]) -> list[dict]:
    """Run a prompt against test cases and check outputs.

    Each test case dict:
      input:             str   — the user message
      expected_format:   'json' | None
      expected_keywords: list[str] — must appear in output (case-insensitive)
      max_length:        int | None
    """
    results = []
    for i, tc in enumerate(test_cases):
        output = call_model(system_prompt, tc["input"])
        checks = {}

        if tc.get("expected_format") == "json":
            try:
                json.loads(output)
                checks["json_valid"] = True
            except json.JSONDecodeError:
                checks["json_valid"] = False

        if tc.get("expected_keywords"):
            output_lower = output.lower()
            for kw in tc["expected_keywords"]:
                checks[f"has_{kw}"] = kw.lower() in output_lower

        if tc.get("max_length"):
            checks["length_ok"] = len(output) <= tc["max_length"]

        passed = all(checks.values()) if checks else True
        results.append({"case": i + 1, "passed": passed, "checks": checks, "output": output[:200]})
        time.sleep(0.5)

    return results

def strip_json_fences(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else text
        text = text.rsplit("```", 1)[0]
    return text.strip()

def evaluate_prompt(system_prompt: str, test_cases: list[dict]) -> list[dict]:
    results = []
    for i, tc in enumerate(test_cases):
        raw_output = call_model(system_prompt, tc["input"])
        output = strip_json_fences(raw_output)  # defensive unwrap
        checks = {}

        if tc.get("expected_format") == "json":
            try:
                json.loads(output)
                checks["json_valid"] = True
            except json.JSONDecodeError:
                checks["json_valid"] = False

        if tc.get("expected_keywords"):
            output_lower = output.lower()
            for kw in tc["expected_keywords"]:
                checks[f"has_{kw}"] = kw.lower() in output_lower

        if tc.get("max_length"):
            checks["length_ok"] = len(output) <= tc["max_length"]

        passed = all(checks.values()) if checks else True
        results.append({"case": i + 1, "passed": passed, "checks": checks, "output": output[:200]})
        time.sleep(0.5)

    return results



## Step 8 — Run the evaluation

Run the harness against your extraction prompt with 5 test cases. If any fail, iterate on the prompt and re-run — each iteration is a version.

**Observe** (answer in a new cell below): name one *specific* test case that failed or nearly failed, and what the automated check caught that eyeballing in Step 4 would have missed. If all 5 passed on the first run, your test cases are probably too easy — add a harder one (an unusual format, a missing field, a tricky edge) and say what you changed. "Automated testing is good" is not an answer; point to a case.

In [17]:
extraction_test_cases = [
    # 1. Tech role with a clear salary range
    {
        "input": "Senior Python Developer at TechCorp, fully remote. $130-160K. Requirements: Python, PostgreSQL, Docker, CI/CD experience.",
        "expected_format": "json",
        "expected_keywords": ["Python", "TechCorp"],
    },
    # 2. No salary listed — tests null handling
    {
        "input": "Hiring: Junior Frontend Engineer, NYC office. Salary not listed. Must know React and TypeScript.",
        "expected_format": "json",
        "expected_keywords": ["React", "TypeScript"],
    },
    # 3. Data/science role
    {
        "input": "Data Scientist — BioPharma Inc, Boston/Remote hybrid. $95-120K base + equity. PhD preferred. Python, R, ML frameworks.",
        "expected_format": "json",
        "expected_keywords": ["BioPharma", "Python"],
    },
    # 4. Non-engineering role — tests schema flexibility
    {
        "input": "Marketing Manager needed at BrightWave Media, Chicago (hybrid, 3 days in office). $70-85K. Skills: SEO, content strategy, Google Analytics.",
        "expected_format": "json",
        "expected_keywords": ["BrightWave", "SEO"],
    },
    # 5. Senior role with equity/tricky compensation phrasing — hard edge case
    {
        "input": (
            "Staff Engineer @ Quantify Labs (Berlin or remote within EU). "
            "Compensation: base salary DOE + significant equity (0.1-0.3%) + annual bonus target 15%. "
            "No fixed range published — will discuss based on experience. "
            "Tech stack: Go, Kubernetes, gRPC, distributed systems background required."
        ),
        "expected_format": "json",
        "expected_keywords": ["Go", "Kubernetes"],
    },
]

results = evaluate_prompt(extract_prompt, extraction_test_cases)
print_eval_results(results)


Results: 5/5 passed

  Case 1: PASS

  Case 2: PASS

  Case 3: PASS

  Case 4: PASS

  Case 5: PASS



---

## Your turn

Draft a system prompt for your capstone's primary interaction pattern. Pick the one thing your system does most — answering questions, classifying inputs, extracting data, generating content — and write a production-grade system prompt for it.

Include: role, task, constraints, output format, one example. Then define at least 3 test cases and run them through the evaluation harness.

When you're done, you should be able to answer two questions:
1. **"Why is each instruction in your prompt necessary?"** — Remove any instruction and describe what would break.
2. **"What would you change if the requirements changed?"** — If the output format changed, or a new category was added, which parts of your prompt would you modify?

In [16]:
# Your capstone system prompt
capstone_prompt = """<role>
You are a requirements extraction engine for an automated workflow deployment pipeline. You read meeting transcripts and convert spoken workflow requirements into a structured deployment spec.
</role>

<task>
Given a raw meeting transcript (from Fathom), extract every distinct automation workflow that was discussed or requested, and return them as a JSON array. Each transcript may describe zero, one, or multiple workflows.
</task>

<schema>
[
  {
    "workflow_name": string,            // short descriptive name, e.g. "New lead to MailerLite"
    "trigger": string,                  // what starts the workflow, e.g. "New WordPress form submission"
    "actions": string[],                // ordered list of steps the workflow performs
    "platforms": string[],              // systems involved, from: ["Make", "MailerLite", "WordPress", "Gmail", "Google Drive", "Other"]
    "confidence": "explicit" | "inferred",  // "explicit" if directly stated as a requirement, "inferred" if pieced together from context
    "open_questions": string[]          // anything ambiguous/unstated that a human should confirm before deployment (empty list if none)
  }
]
</schema>

<constraints>
- Only extract workflows that were actually discussed as something to build — do not invent workflows from tangential small talk.
- If a requirement is vague (e.g. "something like a welcome email"), still extract it, mark "confidence": "inferred", and list the missing specifics in "open_questions" rather than guessing at concrete values.
- Do not auto-fill technical details (API names, field mappings, exact email copy) that were not mentioned in the transcript — that risks the deploy step shipping something nobody actually asked for.
- If the transcript contains no workflow requirements at all, return an empty array: []
</constraints>

<output_format>
Return ONLY the JSON array. No markdown code fences, no explanation, no surrounding text. The first character of your response must be [ and the last character must be ].
</output_format>

<example>
Input: "So when someone fills out the contact form on the site, let's get them added to the MailerLite newsletter list automatically. Oh and also — not now, but eventually — it'd be nice if we could auto-post the podcast episode to the blog once it's edited."

Output: [{"workflow_name": "Contact form to MailerLite", "trigger": "WordPress contact form submission", "actions": ["Add submitter to MailerLite newsletter list"], "platforms": ["WordPress", "MailerLite"], "confidence": "explicit", "open_questions": []}, {"workflow_name": "Podcast episode to blog auto-post", "trigger": "Podcast episode finishes editing", "actions": ["Publish episode as blog post"], "platforms": ["WordPress", "Other"], "confidence": "inferred", "open_questions": ["Not confirmed as an active build — mentioned as a future/maybe item, not a current requirement.", "No detail on what triggers 'finishes editing' (manual upload? specific folder?)", "No detail on blog post format (audio embed, transcript, show notes?)"]}]
</example>"""

# Your test cases
capstone_test_cases = [
    # 1. Clear single explicit workflow
    {
        "input": "Every time we get a new client inquiry through the Elementor form, I want it to create a card in our project tracker and send me a Slack-style notification. Let's use Make for this.",
        "expected_format": "json",
        "expected_keywords": ["Make", "project tracker"],
    },
    # 2. Multiple workflows in one transcript
    {
        "input": "Two things: first, when a Kollel member submits the monthly stipend form, add a row to the Airtable base. Second, once a week, pull the Airtable data and email a summary PDF to the treasurer.",
        "expected_format": "json",
        "expected_keywords": ["Airtable", "treasurer"],
    },
    # 3. No workflow discussed at all — tests the empty-array path
    {
        "input": "Thanks everyone for joining, let's push our next sync to Thursday since Dovid is out this week. Also happy Rosh Chodesh!",
        "expected_format": "json",
        "expected_keywords": [],
    },
    # 4. Vague/inferred requirement — tests confidence + open_questions handling
    {
        "input": "It'd be great if new subscribers got some kind of welcome sequence eventually, haven't fully thought it through yet.",
        "expected_format": "json",
        "expected_keywords": ["inferred", "open_questions"],
    },
]

# Uncomment to run:
results = evaluate_prompt(capstone_prompt, capstone_test_cases)
print_eval_results(results)

Results: 4/4 passed

  Case 1: PASS

  Case 2: PASS

  Case 3: PASS

  Case 4: PASS



---

## Success criteria

You're done when:

- [ ] You have system prompts for 3 task types: classification, extraction, reasoning
- [ ] Classification prompt iterated from v1 (vague) to v2 (precise) to v3 (few-shot) with observable improvement
- [ ] Extraction prompt forces JSON output that parses with `json.loads()` on all test inputs
- [ ] Chain-of-thought applied to reasoning task with documented effect on accuracy
- [ ] Evaluation harness runs 5 test cases and reports pass/fail
- [ ] Capstone system prompt drafted with design rationale
- [ ] "What I learned" cell completed with substantive reflection

## Quality checklist (Best Practices Ownership)

- [ ] Every prompt has role, task, constraints, and output format — no vague requests
- [ ] Few-shot examples include at least one edge case that demonstrates ambiguity resolution
- [ ] CoT applied only where reasoning improves accuracy, not cargo-culted
- [ ] JSON output validated programmatically, not eyeballed
- [ ] Eval harness checks are specific enough to catch real failures but not so strict they produce false negatives
- [ ] You can explain (in your reflection) why you chose each technique for its task type

## What I learned

Write 3–6 bullets in your own words, covering:

- What surprised you
- One thing you'd do differently in production
- Where this connects to your capstone

_This cell is your end-of-exercise reflection, in your own words. It's also what makes this notebook a portfolio artifact when you publish to GitHub._

##What I learned


1.   Prompt instructions alone don't guarantee format compliance — the model still added markdown fences like '''json despite being told not to.
2.   Automated checks catch failures that just reading misses, like valid-looking JSON that fails to actually parse.
3.  This matters directly for my capstone, since the extraction step feeds an auto-deploy pipeline where a silent format failure is costly.
